# TCGA-BRCA Patient Treatment Grouping V1 Review

This notebook reviews the saved TCGA-BRCA provisional patient-level treatment grouping v1 outputs from disk only.
It does not rerun the grouping workflow, reread raw treatment tables, normalize drug names,
freeze treatment arms, or perform modeling.


In [ ]:
from pathlib import Path
import json

import pandas as pd
from IPython.display import display


def detect_repo_root(start_path: Path) -> Path:
    for candidate in [start_path, *start_path.parents]:
        if (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('Unable to locate the repository root from the notebook path.')


def read_tsv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)


repo_root = detect_repo_root(Path.cwd())
latest_pointer_path = (
    repo_root
    / '01-data'
    / 'audit'
    / 'tcga-brca'
    / 'treatment-prep'
    / 'tcga_brca_patient_treatment_grouping_v1_latest.json'
)
if not latest_pointer_path.exists():
    raise FileNotFoundError(
        f'Latest patient treatment grouping v1 pointer not found: {latest_pointer_path}. '
        'Run script 21 first.'
    )

latest_pointer = json.loads(latest_pointer_path.read_text(encoding='utf-8'))
grouping_path = repo_root / latest_pointer['patient_treatment_grouping_v1_tsv']
arm_summary_path = repo_root / latest_pointer['patient_treatment_grouping_v1_arm_summary_tsv']
conflict_path = repo_root / latest_pointer['patient_treatment_grouping_v1_conflict_audit_tsv']
spec_path = repo_root / latest_pointer['patient_treatment_grouping_v1_spec_tsv']
summary_path = repo_root / latest_pointer['patient_treatment_grouping_v1_summary_tsv']
run_log_path = repo_root / latest_pointer['run_log_json']

for required_path in [grouping_path, arm_summary_path, conflict_path, spec_path, summary_path, run_log_path]:
    if not required_path.exists():
        raise FileNotFoundError(f'Required grouping artifact not found: {required_path}')

grouping_df = read_tsv(grouping_path)
arm_summary_df = read_tsv(arm_summary_path)
conflict_df = read_tsv(conflict_path)
spec_df = read_tsv(spec_path)
summary_df = read_tsv(summary_path)
run_log = json.loads(run_log_path.read_text(encoding='utf-8'))

if not bool(run_log.get('validation', {}).get('passed', False)):
    raise ValueError('run_log.json does not report validation.passed == true.')
if grouping_df.empty:
    raise ValueError('patient_treatment_grouping_v1.tsv contains no rows.')

results_root = (
    repo_root
    / '09-trials'
    / '01-tcga-only-source-audited'
    / '05-results'
)
results_root.mkdir(parents=True, exist_ok=True)

approved_groups = [
    'no_drug_record',
    'single_chemotherapy',
    'single_hormone_therapy',
    'single_targeted_therapy',
    'single_immunotherapy',
    'single_ancillary_or_other',
    'mixed_multi_type',
    'missing_type_only',
]

print(f"Run ID    : {latest_pointer['patient_treatment_grouping_v1_run_id']}")
print(f"Profile ID: {latest_pointer['patient_treatment_profile_v1_run_id']}")
print(f"OS ep ID  : {latest_pointer['os_endpoint_v1_run_id']}")
print(f"Pointer   : {latest_pointer_path}")


In [ ]:
# --- Write review tables to 05-results/ ---

review_grouping_path = results_root / '116_patient_treatment_grouping_v1.tsv'
review_arm_summary_path = results_root / '117_patient_treatment_grouping_v1_arm_summary.tsv'
review_conflict_path = results_root / '118_patient_treatment_grouping_v1_conflict_audit.tsv'
review_spec_path = results_root / '119_patient_treatment_grouping_v1_spec.tsv'
review_summary_path = results_root / '120_patient_treatment_grouping_v1_summary.tsv'

grouping_df.to_csv(review_grouping_path, sep='\t', index=False)
arm_summary_df.to_csv(review_arm_summary_path, sep='\t', index=False)
conflict_df.to_csv(review_conflict_path, sep='\t', index=False)
spec_df.to_csv(review_spec_path, sep='\t', index=False)
summary_df.to_csv(review_summary_path, sep='\t', index=False)

print(f'Saved: {review_grouping_path}')
print(f'Saved: {review_arm_summary_path}')
print(f'Saved: {review_conflict_path}')
print(f'Saved: {review_spec_path}')
print(f'Saved: {review_summary_path}')


In [ ]:
# --- Pointer, validation, and key counts ---

print('=== Latest pointer ===')
display(pd.DataFrame([latest_pointer]))

validation = run_log.get('validation', {})
print('\n=== Validation ===')
display(pd.DataFrame([{'check': k, 'value': str(v)} for k, v in validation.items()]))

print('\n=== Key counts ===')
display(pd.DataFrame([{'metric': k, 'value': str(v)} for k, v in run_log.get('counts', {}).items()]))

print('\n=== Saved summary TSV ===')
display(summary_df)


In [ ]:
# --- Group distribution and zero-count groups ---

group_distribution_df = (
    grouping_df['treatment_group_v1']
    .value_counts()
    .reindex(approved_groups, fill_value=0)
    .rename_axis('treatment_group_v1')
    .reset_index(name='patient_count')
)
group_distribution_df['patient_fraction_of_cohort'] = (
    group_distribution_df['patient_count'] / len(grouping_df)
).round(4)

arm_summary_display_df = arm_summary_df.copy()
arm_summary_display_df['group_order'] = arm_summary_display_df['group_order'].astype(int)
arm_summary_display_df = arm_summary_display_df.sort_values('group_order').reset_index(drop=True)

print('=== Group distribution reconstructed from patient table ===')
display(group_distribution_df)

print('\n=== Saved arm summary ===')
display(arm_summary_display_df)


In [ ]:
# --- Manual-review burden and coverage by group ---

manual_review_by_group_df = (
    grouping_df
    .groupby(['treatment_group_v1', 'treatment_group_v1_requires_manual_review'], as_index=False)
    .size()
    .rename(columns={'size': 'patient_count'})
)
manual_review_by_group_df['treatment_group_v1'] = pd.Categorical(
    manual_review_by_group_df['treatment_group_v1'],
    categories=approved_groups,
    ordered=True,
)
manual_review_by_group_df = manual_review_by_group_df.sort_values(
    ['treatment_group_v1', 'treatment_group_v1_requires_manual_review']
).reset_index(drop=True)

coverage_by_group_df = arm_summary_display_df[
    [
        'treatment_group_v1',
        'radiation_overlap_count',
        'radiation_overlap_fraction_within_group',
        'regimen_context_count',
        'regimen_context_fraction_within_group',
        'treatment_timing_count',
        'treatment_timing_fraction_within_group',
    ]
]

print('=== Manual-review burden by group ===')
display(manual_review_by_group_df)

print('\n=== Radiation, regimen-context, and timing coverage by group ===')
display(coverage_by_group_df)


In [ ]:
# --- Conflict audit preview ---

conflict_preview_columns = [
    'bcr_patient_barcode',
    'treatment_group_v1',
    'conflict_type',
    'review_priority',
    'dominant_therapy_type_if_any',
    'drug_therapy_type_single_or_mixed',
    'source_evidence_summary',
]

print('=== Conflict audit preview ===')
display(conflict_df[conflict_preview_columns].head(25))

print('\n=== Spec preview ===')
display(spec_df.head(20))
